# MedAssist AI — Medical RAG Chatbot with DeepSeek + LangChain

A portfolio-ready medical-information chatbot built with **DeepSeek, LangChain, LangGraph, FAISS, Hugging Face embeddings, and medical safety guardrails**.

> **Safety:** This is an educational/portfolio application. It is not a medical device and must not be used to diagnose, prescribe, or manage emergencies.


## 1. Install dependencies

Run once in a fresh environment.

In [ ]:
%pip install -qU langchain langchain-openai langchain-community langchain-huggingface langchain-text-splitters langgraph faiss-cpu sentence-transformers python-dotenv pypdf

## 2. Configure the DeepSeek API key

Create a `.env` file next to the notebook:

```text
DEEPSEEK_API_KEY=your_deepseek_api_key_here
```

Never commit your real API key to Git.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass("Enter your DeepSeek API key: ")

print("DeepSeek API key loaded:", bool(os.getenv("DEEPSEEK_API_KEY")))

In [ ]:
from typing import TypedDict, List, Any
import re

from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END

DEEPSEEK_BASE_URL = "https://api.deepseek.com"
DEEPSEEK_MODEL = "deepseek-chat"

llm = ChatOpenAI(
    model=DEEPSEEK_MODEL,
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url=DEEPSEEK_BASE_URL,
    temperature=0.1,
    max_tokens=1200,
)

print("Configured:", DEEPSEEK_MODEL)

## 3. Medical safety layer

The chatbot:
- provides general information rather than a diagnosis;
- does not prescribe or change medication doses;
- detects several emergency-style phrases before normal RAG;
- encourages professional evaluation when appropriate.

This is an application guardrail, not a guarantee of clinical safety.

In [ ]:
EMERGENCY_PATTERNS = [
    r"severe chest pain",
    r"chest pain.*difficulty breathing",
    r"can't breathe",
    r"cannot breathe",
    r"severe difficulty breathing",
    r"blue lips",
    r"unconscious",
    r"passed out",
    r"loss of consciousness",
    r"stroke symptoms",
    r"face drooping",
    r"slurred speech",
    r"weakness.*one side",
    r"severe bleeding",
    r"bleeding.*won't stop",
    r"vomiting blood",
    r"coughing blood",
    r"suicidal",
    r"kill myself",
    r"overdose",
    r"seizure.*not stopping",
]

def emergency_check(text: str) -> bool:
    return any(re.search(pattern, text.lower()) for pattern in EMERGENCY_PATTERNS)

EMERGENCY_RESPONSE = """### 🚨 Possible medical emergency

Your message contains symptoms or circumstances that may require urgent medical attention.

**Please seek emergency medical care now.** If you are in immediate danger, contact your local emergency service or go to the nearest emergency department. Do not rely on this chatbot to assess or manage an emergency.

If you are with someone who is unconscious, having severe breathing difficulty, experiencing severe chest pain, or showing possible stroke symptoms, seek emergency help immediately.
"""

print("Safety layer ready.")

## 4. Demo medical knowledge base

This small corpus is intentionally generic. For a serious deployment, replace it with clinically reviewed, authoritative, legally usable medical documents.

In [ ]:
medical_documents = [
    Document(page_content="""
Fever is a temporary increase in body temperature and can occur with infections
and other conditions. General supportive care can include adequate fluids and rest.
The significance of fever depends on age, duration, associated symptoms, and the
person's medical circumstances. Persistent, severe, or concerning fever should be
evaluated by a healthcare professional.
""", metadata={"source": "Demo Medical Knowledge Base", "topic": "fever"}),

    Document(page_content="""
Cough can have many causes, including viral respiratory infections, allergies,
asthma, reflux, environmental irritants, and other conditions. A cough that is
severe, persistent, associated with breathing difficulty, chest pain, coughing
blood, or significant deterioration needs medical assessment.
""", metadata={"source": "Demo Medical Knowledge Base", "topic": "cough"}),

    Document(page_content="""
Headache has many possible causes, including tension, migraine, dehydration,
illness, sleep disruption, and medication-related factors. A sudden extremely
severe headache, a headache following significant injury, or a headache with
neurological symptoms such as weakness, confusion, fainting, or difficulty speaking
warrants urgent medical assessment.
""", metadata={"source": "Demo Medical Knowledge Base", "topic": "headache"}),

    Document(page_content="""
Dehydration can occur when the body loses more fluid than it takes in. Possible
symptoms include thirst, dry mouth, dizziness, weakness, and reduced urination.
Treatment depends on severity and cause. Severe dehydration or inability to keep
fluids down can require medical care.
""", metadata={"source": "Demo Medical Knowledge Base", "topic": "dehydration"}),

    Document(page_content="""
High blood pressure often has no obvious symptoms, so it is generally identified
through blood pressure measurement rather than symptoms alone. A single elevated
reading does not necessarily establish a diagnosis. Repeated measurements and
professional assessment may be needed. Extremely high readings accompanied by
concerning symptoms require urgent medical evaluation.
""", metadata={"source": "Demo Medical Knowledge Base", "topic": "blood pressure"}),

    Document(page_content="""
Medicines should be taken according to instructions provided by a qualified
healthcare professional or official product information. A chatbot should not
independently prescribe medicines, change doses, or advise stopping prescription
treatment. Medication questions should consider age, allergies, existing conditions,
other medicines, and the exact product.
""", metadata={"source": "Demo Medical Knowledge Base", "topic": "medications"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)
chunks = splitter.split_documents(medical_documents)
print("Knowledge chunks:", len(chunks))

## 5. FAISS vector store with local embeddings

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("FAISS vector store ready.")

## 6. Optional PDF ingestion

Put trusted, legally usable medical PDFs into `medical_docs/`. This cell can add them to the FAISS index.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def add_pdf_folder(folder="medical_docs"):
    folder_path = Path(folder)
    folder_path.mkdir(exist_ok=True)
    pdf_files = list(folder_path.glob("*.pdf"))

    if not pdf_files:
        print("No PDFs found in:", folder_path.resolve())
        return None

    pdf_docs = []
    for pdf in pdf_files:
        docs = PyPDFLoader(str(pdf)).load()
        for doc in docs:
            doc.metadata["source"] = pdf.name
        pdf_docs.extend(docs)

    pdf_chunks = splitter.split_documents(pdf_docs)
    pdf_store = FAISS.from_documents(pdf_chunks, embedding_model)

    print(f"Loaded {len(pdf_files)} PDF(s), created {len(pdf_chunks)} chunks.")
    return pdf_store

# Uncomment after placing trusted PDFs in medical_docs/
# pdf_store = add_pdf_folder()
# if pdf_store:
#     vectorstore.merge_from(pdf_store)
#     retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

## 7. Grounded DeepSeek prompt

In [ ]:
SYSTEM_PROMPT = """
You are MedAssist AI, a cautious medical-information assistant.

Your job is to provide general health information grounded in the supplied context.

Rules:
- Do not diagnose the user.
- Do not claim a symptom proves a disease.
- Do not prescribe medication, change doses, or tell users to stop prescription medicines.
- Do not invent facts or citations.
- If the retrieved context is insufficient, say so explicitly.
- Ask concise follow-up questions when important information is missing.
- Distinguish general information from individualized medical advice.
- Encourage professional medical assessment when symptoms are persistent, severe,
  worsening, unusual, or concerning.
- Never imply that you are a doctor.
- Use plain, calm language.

Retrieved context:
{context}

Conversation history:
{history}

User question:
{question}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

def format_docs(docs):
    if not docs:
        return "No relevant medical context was retrieved."
    return "\n\n".join(
        f"[Source: {d.metadata.get('source', 'Unknown')} | Topic: {d.metadata.get('topic', 'General')}]\n{d.page_content}"
        for d in docs
    )

def format_history(messages):
    if not messages:
        return "No previous conversation."
    return "\n".join(
        f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}"
        for m in messages[-8:]
    )

## 8. LangGraph workflow

**User → Safety check → Emergency response OR RAG retrieval → DeepSeek → Answer**

In [ ]:
class ChatState(TypedDict, total=False):
    question: str
    history: List[Any]
    emergency: bool
    documents: List[Document]
    context: str
    answer: str

def safety_node(state: ChatState):
    return {"emergency": emergency_check(state["question"])}

def route_after_safety(state: ChatState):
    return "emergency" if state.get("emergency") else "retrieve"

def emergency_node(state: ChatState):
    return {"answer": EMERGENCY_RESPONSE}

def retrieve_node(state: ChatState):
    docs = retriever.invoke(state["question"])
    return {"documents": docs, "context": format_docs(docs)}

def answer_node(state: ChatState):
    messages = prompt.format_messages(
        context=state.get("context", ""),
        history=format_history(state.get("history", [])),
        question=state["question"],
    )
    response = llm.invoke(messages)
    return {"answer": response.content}

graph_builder = StateGraph(ChatState)
graph_builder.add_node("safety", safety_node)
graph_builder.add_node("emergency", emergency_node)
graph_builder.add_node("retrieve", retrieve_node)
graph_builder.add_node("answer", answer_node)

graph_builder.add_edge(START, "safety")
graph_builder.add_conditional_edges(
    "safety",
    route_after_safety,
    {"emergency": "emergency", "retrieve": "retrieve"}
)
graph_builder.add_edge("retrieve", "answer")
graph_builder.add_edge("emergency", END)
graph_builder.add_edge("answer", END)

medical_graph = graph_builder.compile()
print("LangGraph workflow compiled.")

## 9. Chat function + conversation memory

In [ ]:
conversation_history = []

def medical_chat(question: str, show_sources=True):
    global conversation_history

    result = medical_graph.invoke({
        "question": question,
        "history": conversation_history
    })

    answer = result["answer"]

    conversation_history.append(HumanMessage(content=question))
    conversation_history.append(AIMessage(content=answer))

    if show_sources and result.get("documents"):
        print("\n--- Retrieved sources ---")
        seen = set()
        for doc in result["documents"]:
            key = (
                doc.metadata.get("source", "Unknown"),
                doc.metadata.get("topic", "General")
            )
            if key not in seen:
                print("•", key[0], "—", key[1])
                seen.add(key)

    return answer

def reset_chat():
    global conversation_history
    conversation_history = []
    print("Conversation memory cleared.")

print("Medical chat function ready.")

## 10. Test a normal medical-information question

In [ ]:
question = "What are some general reasons someone might have a cough?"
print(medical_chat(question))

## 11. Test emergency routing

This should trigger the safety layer instead of a normal RAG answer.

In [ ]:
print(medical_chat("I have severe chest pain and difficulty breathing."))

## 12. Interactive notebook chat

Type `exit` to stop.

In [ ]:
print("MedAssist AI — type 'exit' to stop.")
print("General medical information only; not a diagnosis or substitute for professional care.\n")

while True:
    user_input = input("You: ").strip()

    if user_input.lower() in {"exit", "quit", "q"}:
        print("Session ended.")
        break

    if not user_input:
        continue

    response = medical_chat(user_input)
    print("\nMedAssist AI:")
    print(response)
    print("\n" + "-" * 80 + "\n")

## 13. Suggested test questions

Try:
- `What are common causes of headaches?`
- `Explain dehydration in simple terms.`
- `What general information is available about fever?`
- `What questions should I ask my doctor about high blood pressure?`

Medication questions are intentionally handled conservatively.

## 14. Production hardening

Before real-world use:

- Replace the demo corpus with clinically reviewed and authoritative sources.
- Add source/version/date metadata and citations.
- Add stronger emergency detection and human escalation.
- Add clinical safety evaluation with qualified professionals.
- Add PII detection/redaction and strict privacy controls.
- Do not store identifiable health conversations by default.
- Add authentication, authorization, rate limits, and audit controls.
- Evaluate retrieval quality, hallucination rate, emergency sensitivity, and refusal behavior.
- Review applicable legal/regulatory requirements before clinical use.

### Resume-ready description

**MedAssist AI — Medical RAG Chatbot | Python, LangChain, LangGraph, DeepSeek, FAISS, Hugging Face**

- Developed a medical-information chatbot using **LangChain + DeepSeek** with FAISS-based RAG for grounded responses from curated medical documents.
- Implemented **LangGraph safety orchestration**, deterministic emergency-intent detection, conversation memory, document retrieval, and controlled LLM generation.
- Built local Sentence Transformer embeddings and PDF ingestion while adding medical guardrails to reduce hallucinations and prevent diagnosis/prescription-style responses.
